# Lab: Who polluted this star?

### Constraining AGB nucleosynthesis with barium stars

**Companion lecture:** *Nucleosynthesis in AGB stars* (`lectures/2_agb_nucleosynthesis.md`) &nbsp;·&nbsp; **Time:** ~30 min

---

AGB stars are hard to observe directly and impossible to take apart. One of the
cleanest ways to test their nucleosynthesis is to look at a star that *ate some
AGB ejecta*.

A **barium star** is a binary. The star we see is an ordinary G/K giant; its
companion is a white dwarf. When that companion was on the AGB it transferred
s-process–rich wind onto the star we now observe. The visible star's surface is
therefore a **mixture**:

$$
X_i^{\rm obs} \;=\; f\,X_i^{\rm AGB\ wind} \;+\; (1-f)\,X_i^{\rm original}
\qquad\qquad
f \;=\; \frac{\Delta M_{\rm accreted}}{\Delta M_{\rm accreted} + M_{\rm env}}
$$

with a single **dilution factor** $f\in[0,1]$: $f\to0$ is heavy dilution (a tiny
bit of wind stirred into a big envelope), $f\to1$ means the envelope is almost
pure accreted material.

**The game:** take a real barium star's measured abundance pattern, and find the
AGB model (initial mass $M$) and dilution $f$ that reproduce it. The recovered
$M$ tells you the mass of the dead white dwarf when it was a star.

This is a stripped-down version of the method in
**Cseh et al. (2018), A&A 620, A146** and
**Cseh et al. (2022), A&A 660, A128**.

### You will
1. see why a raw AGB yield pattern cannot match an observed star, and what dilution does;
2. fit $(M, f)$ for one barium star by $\chi^2$ minimisation, and read the $M$–$f$ degeneracy off the map;
3. classify three more stars by their best-fit AGB companion and connect the result to the physics from the lecture (neutron source, neutron exposure, HBB).

### Data
- **AGB yields:** Karakas & Lugaro (2016) — `labs/yields/KLC25/KL16_z*.dat`. Total ejected mass per element, initial mass 1–8 $M_\odot$, at $Z = 0.007,\,0.014,\,0.03$.
- **Observed stars:** de Castro et al. (2016), MNRAS 459, 4299 — `labs/yields/observed/decastro2016_bastars.csv`. Homogeneous [X/Fe] for 182 barium giants.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# --- locate the repo root (works whether you run from labs/ or the repo root) ---
HERE = Path.cwd()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "labs" / "yields").is_dir())
YIELDS = REPO / "labs" / "yields"
print("repo:", REPO)

## Scaffold — run this, don't edit it

Everything below is provided. The three functions you actually use are:

| function | returns |
|---|---|
| `wind_pattern(Z, M)` | `[X/Fe]` of the **pure** AGB wind (dict), interpolated to mass `M` |
| `dilute(Z, M, f, FeH)` | `[X/Fe]` after mixing wind fraction `f` into a companion of metallicity `FeH` |
| `plot_star(name, models=...)` | plot an observed star, optionally with model curves on top |

`STARS` holds the four stars for this lab. `ELEMENTS` is the list we plot;
`FIT_ELEMENTS` is the subset used in the $\chi^2$.

In [ ]:
# ======================================================================
#  SCAFFOLD  (provided)
# ======================================================================
import re

Z_SUN = 0.0142
Z_GRID = {0.007: "KL16_z007.dat", 0.014: "KL16_z014.dat", 0.03: "KL16_z03.dat"}

# ---- solar composition -------------------------------------------------
# initial_comp.dat stores NUMBER fractions n_i/n_tot (not mass fractions),
# so convert:  X_i = n_i A_i / sum_j n_j A_j .
_ic = pd.read_csv(YIELDS / "initial_comp.dat", sep=r"\s+", header=None,
                  names=["iso", "n", "A"])
_ic["el"] = _ic["iso"].str.extract(r"([a-z]+)")[0]
_ic.loc[_ic["iso"] == "p", "el"] = "h"          # bare proton -> hydrogen
_ic["Xm"] = _ic["n"] * _ic["A"]
_ic["Xm"] /= _ic["Xm"].sum()
SUN = _ic.groupby("el")["Xm"].sum()             # solar mass fractions, by element
#   X_H = 0.715, Y_He = 0.271, Z = 0.014, [Fe] = 1.4e-3  -- looks solar, good.

# ---- AGB yields ------------------------------------------------------
def load_yields(Z):
    """Raw KL16 table for one metallicity: columns m, Zini, mrem, el, Zn, y(=ejected mass)."""
    df = pd.read_csv(YIELDS / "KLC25" / Z_GRID[Z], sep=r"\s+", header=None,
                     names=["m", "Zini", "mrem", "el", "Zn", "y"])
    df.loc[df["Zn"] == 1, "el"] = "h"
    df.loc[df["Zn"] == 2, "el"] = "he"
    return df

def _wind_grid(Z):
    """element x initial-mass table of wind MASS FRACTIONS, with broken models dropped."""
    df = load_yields(Z)
    piv = df.pivot_table(index="el", columns="m", values="y").sort_index(axis=1)
    good = []
    for M in piv.columns:
        col = piv[M]
        env = M - df.loc[df.m == M, "mrem"].iloc[0]
        # a valid model has H+He present and sum(ejecta) ~ (M - M_remnant)
        if col.get("h", 0) > 0 and col.get("he", 0) > 0 and abs(col.sum() / env - 1) < 0.05:
            good.append(M)
        else:
            print(f"  ! dropping broken model  Z={Z}  M={M}")
    piv = piv[good]
    return piv / piv.sum(axis=0)          # normalise each column to a composition

_WIND_CACHE = {Z: _wind_grid(Z) for Z in Z_GRID}

def wind_massfrac(Z, M):
    g = _WIND_CACHE[Z]
    ms = g.columns.values
    return pd.Series({e: np.interp(M, ms, g.loc[e].values) for e in g.index})

def initial_massfrac(FeH):
    """Scaled-solar composition of the companion's original envelope at metallicity FeH."""
    x = SUN.copy()
    metals = ~x.index.isin(["h", "he"])
    x[metals] = SUN[metals] * 10.0**FeH
    hHe = SUN[["h", "he"]] / SUN[["h", "he"]].sum() * (1.0 - x[metals].sum())
    x[["h", "he"]] = hHe
    return x

def _to_XFe(x):
    """mass-fraction Series -> [X/Fe] dict (A cancels in the ratio-of-ratios)."""
    r = (x / x["fe"]) / (SUN / SUN["fe"])
    return {e.capitalize(): float(np.log10(v)) for e, v in r.items() if v > 0}

def wind_pattern(Z, M):
    """[X/Fe] of the pure AGB wind."""
    return _to_XFe(wind_massfrac(Z, M).reindex(SUN.index).fillna(0.0))

def dilute(Z, M, f, FeH):
    """[X/Fe] of a companion envelope after mixing in wind fraction f."""
    w = wind_massfrac(Z, M).reindex(SUN.index).fillna(0.0)
    x0 = initial_massfrac(FeH)
    return _to_XFe(f * w + (1.0 - f) * x0)

def nearest_Z(FeH):
    return min(Z_GRID, key=lambda z: abs(np.log10(z / Z_SUN) - FeH))

# ---- observed stars (de Castro et al. 2016) --------------------------
ELEMENTS     = ["Na", "Mg", "Al", "Si", "Ca", "Ti", "Cr", "Ni", "Y", "Zr", "La", "Ce", "Nd"]
FIT_ELEMENTS = ["Na", "Y", "Zr", "La", "Ce", "Nd"]

# representative 1-sigma abundance errors (de Castro et al. 2016, sect. 4.4 / Table 9)
SIGMA = dict(Na=.12, Mg=.13, Al=.12, Si=.10, Ca=.11, Ti=.13, Cr=.12, Ni=.10,
             Y=.16, Zr=.20, La=.15, Ce=.14, Nd=.16)

STARS = {
    # name        FeH     [X/Fe] for ELEMENTS
    "HD 213084": dict(FeH=-0.09, XFe=dict(Na=0.04, Mg=-0.03, Al=0.05, Si=0.08, Ca=0.07,
                                          Ti=0.00, Cr=0.03, Ni=0.02,
                                          Y=0.80, Zr=0.72, La=1.10, Ce=0.91, Nd=0.89)),
    "HD 215555": dict(FeH=-0.08, XFe=dict(Na=-0.07, Mg=0.06, Al=0.10, Si=0.09, Ca=0.08,
                                          Ti=0.01, Cr=0.03, Ni=-0.07,
                                          Y=0.99, Zr=0.80, La=0.85, Ce=0.88, Nd=0.65)),
    "HD 174204": dict(FeH=-0.06, XFe=dict(Na=0.09, Mg=0.04, Al=0.13, Si=0.22, Ca=0.06,
                                          Ti=-0.01, Cr=-0.01, Ni=0.02,
                                          Y=-0.11, Zr=-0.07, La=0.07, Ce=0.19, Nd=0.13)),
    "HD 123396": dict(FeH=-1.04, XFe=dict(Na=0.07, Mg=0.44, Al=0.28, Si=0.36, Ca=0.30,
                                          Ti=0.14, Cr=-0.03, Ni=0.04,
                                          Y=0.60, Zr=0.73, La=1.55, Ce=1.29, Nd=1.25)),
}

_ZATOM = dict(Na=11, Mg=12, Al=13, Si=14, Ca=20, Ti=22, Cr=24, Ni=28,
              Y=39, Zr=40, La=57, Ce=58, Nd=60)
_XPOS = [_ZATOM[e] for e in ELEMENTS]

def plot_star(name, models=None, ax=None):
    """models: list of (label, XFe_dict) to overlay."""
    s = STARS[name]
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 4))
    y   = [s["XFe"][e] for e in ELEMENTS]
    err = [SIGMA[e]     for e in ELEMENTS]
    ax.errorbar(_XPOS, y, yerr=err, fmt="o", color="k", ms=7, capsize=3,
                zorder=5, label=f"{name}  ([Fe/H]={s['FeH']:+.2f})")
    for label, patt in (models or []):
        ax.plot(_XPOS, [patt.get(e, np.nan) for e in ELEMENTS], "-s", ms=4, alpha=.85, label=label)
    ax.axhspan(-0.1, 0.1, color="0.85", zorder=0)
    ax.set_xticks(_XPOS); ax.set_xticklabels(ELEMENTS)
    ax.set_ylabel("[X/Fe]"); ax.legend(fontsize=9, frameon=False)
    ax.set_title(name)
    return ax

print("scaffold loaded — stars:", ", ".join(STARS))

---
## Task 1 — why you need dilution  *(≈ 7 min)*

`HD 213084` is a strong barium star. First look at it on its own, then throw the
**raw, undiluted** wind of a 2 $M_\odot$ AGB model on top.

In [ ]:
plot_star("HD 213084",
          models=[("pure wind, 2 Msun, Z=0.014", wind_pattern(0.014, 2.0))])
plt.show()

The **shape** is roughly right (flat light elements, a big s-process bump) but the
**amplitude** is way off: the pure wind sits ~1 dex above the star for the heavy
elements. The accreted material was diluted in the companion's envelope.

**Your turn.** Overlay `dilute(0.014, 2.0, f, FeH=-0.09)` for a few values of
`f` between 0 and 1. Roughly what `f` brings the model onto the data?

In [ ]:
# TODO: loop over a few f values and overlay dilute(0.014, 2.0, f, FeH=-0.09)
ax = plot_star("HD 213084")
for f in []:          # <-- put a few values here, e.g. [0.05, 0.2, 0.5]
    ax.plot(_XPOS, [dilute(0.014, 2.0, f, -0.09).get(e, np.nan) for e in ELEMENTS],
            "-", alpha=.7, label=f"f = {f}")
ax.legend(fontsize=9, frameon=False)
plt.show()

---
## Task 2 — fit the companion mass  *(≈ 10 min)*

Now do it properly. Define a reduced $\chi^2$ over `FIT_ELEMENTS`:

$$
\chi^2_\nu(M,f) \;=\; \frac{1}{N}\sum_i
\left(\frac{[X_i/{\rm Fe}]^{\rm obs} - [X_i/{\rm Fe}]^{\rm model}(M,f)}{\sigma_i}\right)^2
$$

Use `Z = nearest_Z(FeH)` for the AGB model (the companion formed from the same
gas as the star we see). Then scan `M` and `f` on a grid and take the minimum.

In [ ]:
def chi2(name, Z, M, f):
    s = STARS[name]
    model = dilute(Z, M, f, s["FeH"])
    # TODO: return reduced chi^2 over FIT_ELEMENTS
    raise NotImplementedError

# --- grid search (fill in chi2 above, then run) ---
name = "HD 213084"
Z    = nearest_Z(STARS[name]["FeH"])
Mgrid = np.arange(1.0, 5.01, 0.25)
fgrid = np.linspace(0.02, 0.98, 60)

G = np.array([[chi2(name, Z, M, f) for f in fgrid] for M in Mgrid])
i, j = np.unravel_index(np.argmin(G), G.shape)
Mbest, fbest = Mgrid[i], fgrid[j]
print(f"{name}:  best M = {Mbest:.2f} Msun,  f = {fbest:.2f},  chi2/N = {G[i,j]:.2f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2))
plot_star(name, models=[(f"best fit: M={Mbest:.2f}, f={fbest:.2f}",
                         dilute(Z, Mbest, fbest, STARS[name]["FeH"]))], ax=a1)
im = a2.pcolormesh(fgrid, Mgrid, np.log10(G), shading="nearest")
a2.plot(fbest, Mbest, "r*", ms=15)
a2.set_xlabel("dilution f"); a2.set_ylabel("AGB initial mass  [Msun]")
a2.set_title(r"$\log_{10}\,\chi^2_\nu$"); fig.colorbar(im, ax=a2)
plt.show()

**Questions**
- What is the best-fit initial mass of the white-dwarf progenitor?
- Look at the $\chi^2$ map: is $M$ or $f$ better constrained? Which observed
  elements would you most want smaller error bars on to tighten $M$?
- The literature ([Fe/H], mass estimates in de Castro et al. 2016) puts these
  progenitors around 1.5–3 $M_\odot$. Does your fit land there?

---
## Task 3 — three more stars  *(≈ 8 min)*

Wrap Task 2 into a function and run it on the other three stars. Then **interpret**.

In [ ]:
def fit_star(name, Mgrid=np.arange(1.0, 5.01, 0.25), fgrid=np.linspace(0.02, 0.98, 60)):
    Z = nearest_Z(STARS[name]["FeH"])
    G = np.array([[chi2(name, Z, M, f) for f in fgrid] for M in Mgrid])
    i, j = np.unravel_index(np.argmin(G), G.shape)
    return dict(Z=Z, M=Mgrid[i], f=fgrid[j], chi2=G[i, j], Mgrid=Mgrid, G=G)

for name in ["HD 215555", "HD 174204", "HD 123396"]:
    r = fit_star(name)
    print(f"{name:12s} [Fe/H]={STARS[name]['FeH']:+.2f}  ->  Z_agb={r['Z']}, "
          f"M={r['M']:.2f}, f={r['f']:.2f}, chi2/N={r['chi2']:.2f}")
    plot_star(name, models=[("best fit", dilute(r["Z"], r["M"], r["f"], STARS[name]["FeH"]))])
    plt.show()

**HD 215555 vs HD 213084** — both are strong barium stars, but HD 215555 has
`[hs/ls] < 0` (Y, Zr enhanced *more* than La, Ce, Nd) while HD 213084 has
`[hs/ls] > 0`. Compare their best-fit masses. Which neutron source
($^{13}$C$(\alpha,n)$ vs $^{22}$Ne$(\alpha,n)$) does each imply, and why does
that follow from the lecture? Why can't a 5 $M_\odot$ model fit *either* star?

**HD 174204** — what value of `f` did the fit want, and what does the pattern
tell you? Is there any evidence this star was polluted at all?

**HD 123396** — `[Fe/H] = -1.04`, well below the model grid (nearest is
$Z=0.007$, i.e. [Fe/H] $\approx -0.3$). Look at the best-fit `f`. Is it
physical for a giant? What does this failure tell you about what kind of models
you'd need — and what class of object ([lecture](../lectures/2_agb_nucleosynthesis.md):
CH stars, CEMP-s stars) lives down there?

---
## Wrap-up

- You recovered AGB companion masses from surface abundances of the *secondary*
  star — the AGB star itself is a white dwarf and was never observed.
- **Degeneracies you hit:** $M$–$f$ (amplitude), and $M$–$Z$–(pocket). The
  heavy/light s ratio `[hs/ls]` and, where available, `[Pb/hs]` are what break
  them — they measure the *neutron exposure*, which is set by the physics, not
  by how much got diluted.
- The single biggest hidden knob in real work is the **$^{13}$C pocket** (mass
  and shape), which these standard KL16 models fix. Cseh et al. (2018, 2022) vary
  it; the highest-`[hs/ls]` stars generally want a bigger pocket than standard.
- Fitting *one* star is dangerous. The method earns its keep on *samples* —
  `labs/yields/observed/decastro2016_bastars.csv` has 180 more.

---
## Optional: do the grid in Fortran

The $\chi^2$ grid is a tight numerical loop — the kind of thing you'd push into
Fortran in a real pipeline. This cell writes a small `.f90`, compiles it with
`gfortran`, and checks it against the Python result. **Needs `gfortran`** (skip
if you don't have it).

In [ ]:
import os, subprocess, shutil, textwrap

# The kernel: mix wind and original composition in MASS FRACTIONS (numerator and
# Fe separately), then form [X/Fe].  windE(nel,nm), windFe(nm) are mass fractions.
FSRC = textwrap.dedent("""
    program chi2grid
      implicit none
      integer, parameter :: nel = 6
      integer :: nm, nf, i, j, k, io
      real(8) :: obs(nel), sig(nel), x0E(nel), solr(nel), x0Fe
      real(8) :: f, xe, xfe, model, c2
      real(8), allocatable :: mgrid(:), fgrid(:), windE(:,:), windFe(:)
      open(newunit=io, file="chi2grid.in", status="old", action="read")
      read(io,*) nm, nf
      allocate(mgrid(nm), fgrid(nf), windE(nel,nm), windFe(nm))
      read(io,*) mgrid
      read(io,*) fgrid
      read(io,*) obs
      read(io,*) sig
      read(io,*) solr          ! solar X_i/X_Fe for the nel fit elements
      read(io,*) x0E           ! original-envelope mass fractions, nel elements
      read(io,*) x0Fe          ! original-envelope Fe mass fraction
      do i = 1, nm
         read(io,*) windE(:,i)
      end do
      read(io,*) windFe
      close(io)
      do i = 1, nm
         do j = 1, nf
            f  = fgrid(j)
            c2 = 0.d0
            do k = 1, nel
               xe    = f*windE(k,i) + (1.d0-f)*x0E(k)
               xfe   = f*windFe(i)  + (1.d0-f)*x0Fe
               model = log10( (xe/xfe) / solr(k) )
               c2    = c2 + ((obs(k) - model) / sig(k))**2
            end do
            write(*,'(3(1x,es16.8))') mgrid(i), f, c2/nel
         end do
      end do
    end program chi2grid
""")

def _run_fortran_demo():
    if shutil.which("gfortran") is None:
        print("gfortran not found — skipping (this cell is optional).")
        return
    import tempfile
    work = Path(tempfile.mkdtemp())          # build artifacts stay out of labs/
    src, exe, inp = work / "chi2grid.f90", work / "chi2grid", work / "chi2grid.in"
    src.write_text(FSRC)
    env = dict(os.environ)
    def build():
        subprocess.run(["gfortran", "-O2", str(src), "-o", str(exe)],
                       check=True, capture_output=True, text=True, env=env)
    try:
        build()
    except subprocess.CalledProcessError:
        # common macOS/Homebrew gfortran issue: "ld: library not found for -lSystem"
        sdk = subprocess.run(["xcrun", "--show-sdk-path"], capture_output=True, text=True).stdout.strip()
        env["LIBRARY_PATH"] = f"{sdk}/usr/lib"
        build()

    name = "HD 213084"; s = STARS[name]; Z = nearest_Z(s["FeH"])
    Mg = np.arange(1.0, 5.01, 0.25); fg = np.linspace(0.02, 0.98, 60)
    solr = [float(SUN[e.lower()] / SUN["fe"]) for e in FIT_ELEMENTS]
    x0v  = initial_massfrac(s["FeH"])
    x0E  = [float(x0v[e.lower()]) for e in FIT_ELEMENTS]
    x0Fe = float(x0v["fe"])
    windE, windFe = [], []
    for M in Mg:
        w = wind_massfrac(Z, M).reindex(SUN.index).fillna(0.0)
        windE.append([float(w[e.lower()]) for e in FIT_ELEMENTS])
        windFe.append(float(w["fe"]))
    obs = [s["XFe"][e] for e in FIT_ELEMENTS]
    sig = [SIGMA[e]     for e in FIT_ELEMENTS]

    def row(v): return " ".join(f"{x:.8e}" for x in v) + "\n"
    with open(inp, "w") as fo:
        fo.write(f"{len(Mg)} {len(fg)}\n")
        fo.write(row(Mg)); fo.write(row(fg)); fo.write(row(obs)); fo.write(row(sig))
        fo.write(row(solr)); fo.write(row(x0E)); fo.write(f"{x0Fe:.8e}\n")
        for r in windE: fo.write(row(r))
        fo.write(row(windFe))

    out = subprocess.run([str(exe)], cwd=work, capture_output=True, text=True, check=True).stdout
    arr = np.array([ln.split() for ln in out.strip().splitlines()], float)
    k = arr[:, 2].argmin()
    print(f"Fortran best:  M = {arr[k,0]:.2f},  f = {arr[k,1]:.2f},  chi2/N = {arr[k,2]:.3f}")
    print("Should match your Python Task-2 result for HD 213084.")

try:
    _run_fortran_demo()
except subprocess.CalledProcessError as e:
    print("gfortran present but the build/run failed (toolchain issue, not the lab):")
    print((e.stderr or "").strip()[:500])

### Stretch: the classical single-exposure s-process

The fit told you a *neutron exposure* indirectly, through `[hs/ls]`. You can get
it directly from the classical model. For a single exponential exposure
$\rho(\tau) = (N_0/\tau_0)\,e^{-\tau/\tau_0}$, the s-only abundances obey

$$
\sigma_A N_A \;=\; \frac{N_0}{\tau_0}\prod_{k=1}^{A}\left(1+\frac{1}{\sigma_k\,\tau_0}\right)^{-1}
$$

where $\sigma_A$ are the 30 keV Maxwellian-averaged $(n,\gamma)$ cross sections.

Write a short Fortran (or Python) routine that evaluates this recursion along the
s-path for a range of $\tau_0$, forms `[hs/ls]` (e.g. La/Y) and `[Pb/hs]`, and
finds the $\tau_0$ that matches HD 213084. Compare it to the effective exposure
of your best-fit KL16 model. Cross sections: KADoNiS (`kadonis.org`), or the
compilation in Käppeler et al. (2011), RMP 83, 157.